# VisWord 04 — Training ladder (rows 7–15 + 22–24)

This notebook trains the *fine-tuned* rows of the method ladder. Key rows:

| # | Backbone | Trainable | Head | Loss |
|---|---|---|---|---|
| 7 | DINOv2 ViT-B/14 | head only (frozen) | linear 768→256 | MultiSim |
| 8 | DINOv2 ViT-B/14 | last 2 blocks | MLP | MultiSim |
| 9 | DINOv2 ViT-B/14 | last 4 blocks | MLP | MultiSim |
| 10 | DINOv2 ViT-B/14 | last 4 blocks | MLP | InfoNCE |
| 11 | DINOv2 ViT-B/14 | last 4 blocks | MLP | Triplet |
| 12 | DINOv2 ViT-B/14 | last 4 blocks | **SALAD-full** | MultiSim |
| 22 | **CLIP-ViT-B/16 image** | last 4 blocks | MLP | MultiSim |
| 23 | **CLIP-ViT-B/16 image** | last 4 blocks | **SALAD-full** | MultiSim |

Uses VALAR's training loop (`visword.train:main`) with a config YAML per row.

**Runtime:** A100 strongly recommended (L4 OK, T4 slow).  **Wallclock per row:** ~30–60 min on A100 at 10k train, 3 ep, bs=16.

In [ ]:
# Session bootstrap
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, yaml, subprocess
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
os.environ['RUNS_DIR'] = f'{PROJECT}/runs'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Pick training scale

In [ ]:
import json
cache = Path(os.environ['DATA_DIR']) / 'wiki_ss'
manifest = json.load(open(cache / 'manifest.json'))
print('cache has', manifest['num_rows'], 'rows')

NUM_TRAIN = min(manifest['num_rows'] - 1000, 50_000)   # leave 1000 for eval
NUM_EVAL = 1000
EPOCHS = 3
BATCH_SIZE = 16       # 32 OOMs on T4 for SALAD; A100 can handle 64+
print(f'will train on {NUM_TRAIN} pages × {EPOCHS} epochs, eval on {NUM_EVAL}, bs={BATCH_SIZE}')

## Helper — write a ladder config YAML + invoke `visword.train:main`

In [ ]:
def make_config(row_id, label, model_kind, salad_ablation='full', num_trainable_blocks=4, loss='multisim'):
    cfg = {
        'experiment_name': f'visword-{row_id}-{label}',
        'model_kind': model_kind,
        'data': {'num_train_samples': NUM_TRAIN, 'num_eval_samples': NUM_EVAL},
        'backbone': {'num_trainable_blocks': num_trainable_blocks},
        'salad': {'ablation': salad_ablation},
        'train': {
            'loss': loss, 'k_per_page': 4, 'batch_size': BATCH_SIZE,
            'epochs': EPOCHS, 'lr_backbone': 1e-5, 'lr_head': 5e-4,
            'eval_every_steps': 250, 'num_diag_batches': 3,
        },
        'eval': {'k_values': [1,5,10,20], 'phase1_max_pages': NUM_EVAL, 'phase2_max_queries': 200},
    }
    cfg_path = Path('configs') / f'colab_{row_id}_{label}.yaml'
    yaml.safe_dump(cfg, open(cfg_path, 'w'))
    return cfg_path

def run_training(row_id, label, **kwargs):
    cfg_path = make_config(row_id, label, **kwargs)
    run_name = f'{row_id}_{label}'
    cmd = ['python', '-u', '-m', 'visword.train', '--config', str(cfg_path), '--run-name', run_name]
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    # eval follows
    run_dirs = sorted(Path(PROJECT, 'runs').glob(f'*_{run_name}_*'))
    run_dir = run_dirs[-1]
    subprocess.run(['python', '-u', '-m', 'visword.eval_phase1', '--run-dir', str(run_dir)],
                   check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    subprocess.run(['python', '-u', '-m', 'visword.eval_phase2', '--run-dir', str(run_dir)],
                   check=True, env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'})
    return run_dir

## Row 9 — DINOv2 + MLP head, last 4 blocks (the CLS baseline)

In [ ]:
run_training('row09', 'cls_main', model_kind='cls', num_trainable_blocks=4, loss='multisim')

## Row 12 — DINOv2 + SALAD-full (the headline SALAD run)

In [ ]:
run_training('row12', 'salad_main', model_kind='salad', salad_ablation='full', loss='multisim')

## Row 7 — Linear probe

In [ ]:
# Linear probe = frozen backbone (num_trainable_blocks=0) + single linear head.
# Our current config has MLP head; for a true linear probe you'd need to patch
# DINOv2CLS.head → nn.Linear. Here we approximate with frozen-backbone MLP.
run_training('row07', 'linear_probe', model_kind='cls', num_trainable_blocks=0, loss='multisim')

## Row 8 — Last 2 blocks fine-tune

In [ ]:
run_training('row08', 'cls_last2', model_kind='cls', num_trainable_blocks=2, loss='multisim')

## Row 10 — InfoNCE loss

In [ ]:
run_training('row10', 'cls_infonce', model_kind='cls', num_trainable_blocks=4, loss='infonce')

## Row 11 — Triplet loss

In [ ]:
run_training('row11', 'cls_triplet', model_kind='cls', num_trainable_blocks=4, loss='triplet')

## Rows 22–23 — CLIP image backbone + MLP / SALAD heads (optional, requires a backbone refactor)

These need a new `CLIPImageBackbone` wrapper that replaces `OfficialDINOv2` in `dinov2_cls.py` / `dinov2_salad.py`. Below is a minimal sketch that subclasses our `DINOv2CLS` / `DINOv2SALAD` but swaps the backbone.

Skip if time-constrained — DINOv2 rows are the more critical experiments.

In [ ]:
# Minimal CLIP-image + SALAD sketch (training implemented via a quick script).
# If you want to run it: write a tiny train_clip_salad.py under repo root that
# replaces `self.backbone = OfficialDINOv2(...)` with `self.backbone = CLIPImageBackbone(...)`.
print('CLIP fine-tune rows (22-24) require a small backbone refactor; see plan Part D.')

## Final summary of trained rows

In [ ]:
import pandas as pd
rows_out = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row0[7-9]_*')) + \
         list(Path(f'{PROJECT}/runs').glob('*_row1[0-2]_*')):
    try:
        p1 = json.load(open(d / 'phase1_recall.json'))
        p2 = json.load(open(d / 'phase2_recall.json'))
        rows_out.append({
            'row': d.name.split('_row')[1][:2],
            'label': d.name.split('_', 2)[-1],
            'P1_R@10': p1['recall']['10'],
            'P2_R@1': p2['recall']['1'],
        })
    except Exception:
        pass
df = pd.DataFrame(rows_out)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/training_ladder_summary.csv', index=False)